In [ ]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
# from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
import gc
from VideoSwinTransformer_probe import SwinTransformer3D, VSTFCAFLinearProbe
from einops import rearrange
from tqdm import tqdm
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="r2plus1d+VST + AST fusion after transformer blocks"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()


In [2]:

logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:2' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-08 22:07:30,446]::2853905153::DEBUG::=============r2plus1d+VST + AST fusion after transformer blocks+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:2


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)

1


In [4]:
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

In [5]:
backbone_checkpoint_path = './save_swintransformer_folder/model_best_multiseed.pth'
probe_best_model_path = './save_swintransformer_folder/vst_fccc_linear_probe_best.pth'
os.makedirs(os.path.dirname(probe_best_model_path), exist_ok=True)
print(1)


1


In [6]:
validation_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/valid_set.dat","valid")
print(1)

1


In [7]:
train_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/train_set.dat","train")
print(1)

1


In [8]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [9]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [10]:
num_workerssz = 20
lr = 3e-05
epochs = 50
print(1)

1


In [11]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0], fullshot=train_set_data[1], tagdata=train_set_data[2], transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0], fullshot=validation_set_data[1], tagdata=validation_set_data[2], transform=transform)
print(1)


1


In [12]:
criterion = torch.nn.L1Loss().to(device)
print(1)


1


In [13]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [14]:
# SEEDS = [42, 123, 777, 1004, 2024]
SEEDS = [42]

patience = 10

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def load_backbone_checkpoint(model, checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    state_dict = checkpoint['model'] if isinstance(checkpoint, dict) and 'model' in checkpoint else checkpoint
    model.load_state_dict(state_dict, strict=True)
    return model

seed_results = []
global_best_mean_mae = float('inf')
global_best_seed = None
global_best_epoch = None

with torch.cuda.device(2):

    for seed_idx, seed in enumerate(SEEDS, start=1):

        logger.info(
            '================ Probe Seed {}/{}: {} ================'.format(
                seed_idx, len(SEEDS), seed
            )
        )

        set_seed(seed)

        train_generator = torch.Generator()
        train_generator.manual_seed(seed)

        val_generator = torch.Generator()
        val_generator.manual_seed(seed + 10000)

        train_dataloader = DataLoader(
            dataset=train_set_data,
            batch_size=batchsz,
            shuffle=True,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=train_generator
        )

        val_dataloader = DataLoader(
            dataset=val_set_data,
            batch_size=batchsz,
            shuffle=False,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=val_generator
        )

        # ----------------------------------------------------
        # Frozen trained backbone + freshly initialized probes
        # ----------------------------------------------------
        backbone = SwinTransformer3D()
        backbone = load_backbone_checkpoint(
            backbone,
            backbone_checkpoint_path
        )

        model = VSTFCAFLinearProbe(
            backbone=backbone,
            num_outputs=5
        ).to(device)

        optimizer = torch.optim.AdamW(
            model.probe_heads.parameters(),
            lr=lr
        )

        probe_names = list(model.probe_heads.keys())

        seed_best = {
            name: {
                'mae': float('inf'),
                'epoch': 0
            }
            for name in probe_names
        }

        seed_best_mean_mae = float('inf')
        no_improve_count = 0

        # ----------------------------------------------------
        # Sanity checks
        # ----------------------------------------------------
        logger.info(
            'Backbone trainable params: {}'.format(
                sum(p.numel() for p in model.backbone.parameters() if p.requires_grad)
            )
        )
        logger.info(
            'Probe trainable params: {}'.format(
                sum(p.numel() for p in model.probe_heads.parameters() if p.requires_grad)
            )
        )
        logger.info('Probe dimensions: {}'.format(model.probe_dims))

        for epoch in range(epochs):

            # ==================================================
            # Train probes only
            # ==================================================
            model.train()

            train_abs_sum = {name: 0.0 for name in probe_names}
            train_count = {name: 0 for name in probe_names}

            for audiodata, fullshot, big_five_data in tqdm(
                train_dataloader,
                desc='Probe Seed {} Train Epoch {}'.format(seed, epoch + 1)
            ):
                fullshot = fullshot.permute(0, 4, 1, 2, 3)
                big_five_data = big_five_data.permute(0, 2, 1).squeeze()

                audiodata = audiodata.to(device)
                fullshot = fullshot.to(device)
                big_five_data = big_five_data.to(device)

                optimizer.zero_grad()

                outputs = model(
                    fullshot,
                    audiodata
                )

                losses = {
                    name: criterion(outputs[name], big_five_data)
                    for name in probe_names
                }

                loss = torch.stack(
                    [losses[name] for name in probe_names]
                ).mean()

                loss.backward()
                optimizer.step()

                with torch.no_grad():
                    for name in probe_names:
                        train_abs_sum[name] += (
                            outputs[name] - big_five_data
                        ).abs().sum().item()
                        train_count[name] += big_five_data.numel()

            train_mae = {
                name: train_abs_sum[name] / train_count[name]
                for name in probe_names
            }

            # ==================================================
            # Validation
            # ==================================================
            model.eval()

            val_abs_sum = {name: 0.0 for name in probe_names}
            val_count = {name: 0 for name in probe_names}

            with torch.no_grad():

                for audiodata, fullshot, big_five_data in tqdm(
                    val_dataloader,
                    desc='Probe Seed {} Val Epoch {}'.format(seed, epoch + 1)
                ):
                    fullshot = fullshot.permute(0, 4, 1, 2, 3)
                    big_five_data = big_five_data.permute(0, 2, 1).squeeze()

                    audiodata = audiodata.to(device)
                    fullshot = fullshot.to(device)
                    big_five_data = big_five_data.to(device)

                    outputs = model(
                        fullshot,
                        audiodata
                    )

                    for name in probe_names:
                        val_abs_sum[name] += (
                            outputs[name] - big_five_data
                        ).abs().sum().item()
                        val_count[name] += big_five_data.numel()

            val_mae = {
                name: val_abs_sum[name] / val_count[name]
                for name in probe_names
            }

            mean_val_mae = float(
                np.mean([val_mae[name] for name in probe_names])
            )

            # Each representation keeps its own best epoch/result.
            for name in probe_names:
                if val_mae[name] < seed_best[name]['mae']:
                    seed_best[name]['mae'] = float(val_mae[name])
                    seed_best[name]['epoch'] = epoch + 1

            # Patience is based only on the mean validation MAE across probes.
            if mean_val_mae < seed_best_mean_mae:
                seed_best_mean_mae = mean_val_mae
                no_improve_count = 0

                if mean_val_mae < global_best_mean_mae:
                    global_best_mean_mae = mean_val_mae
                    global_best_seed = seed
                    global_best_epoch = epoch + 1

                    torch.save(
                        {
                            'seed': seed,
                            'epoch': epoch + 1,
                            'probe_model': model.probe_heads.state_dict(),
                            'mean_val_mae': mean_val_mae,
                            'per_probe_val_mae': val_mae,
                            'probe_dims': model.probe_dims
                        },
                        probe_best_model_path
                    )
            else:
                no_improve_count += 1

            print(
                '\nSeed {} | Epoch {:03d} | Mean Val MAE={:.6f} | Patience={}/{}'.format(
                    seed,
                    epoch + 1,
                    mean_val_mae,
                    no_improve_count,
                    patience
                )
            )
            print(
                '{:<22} {:>12} {:>12} {:>12}'.format(
                    'Representation',
                    'Train MAE',
                    'Val MAE',
                    '1-MAE'
                )
            )
            print('-' * 62)

            for name in probe_names:
                print(
                    '{:<22} {:>12.6f} {:>12.6f} {:>12.6f}'.format(
                        name,
                        train_mae[name],
                        val_mae[name],
                        1.0 - val_mae[name]
                    )
                )

            if no_improve_count >= patience:
                logger.info(
                    'Probe early stopping: seed={} at epoch {}. '
                    'Best mean val MAE={:.6f}'.format(
                        seed,
                        epoch + 1,
                        seed_best_mean_mae
                    )
                )
                break

            torch.cuda.empty_cache()

        seed_results.append(
            {
                'seed': seed,
                'representations': {
                    name: {
                        'best_epoch': seed_best[name]['epoch'],
                        'best_mae': seed_best[name]['mae'],
                        'best_1_mae': 1.0 - seed_best[name]['mae']
                    }
                    for name in probe_names
                }
            }
        )

        del model
        del backbone
        del optimizer
        del train_dataloader
        del val_dataloader

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()


# ============================================================
# Final multi-seed probe statistics
# ============================================================
print('\n' + '=' * 94)
print('FINAL VST-FCCC LAYERWISE LINEAR PROBE RESULTS')
print('=' * 94)
print(
    '{:<22} {:>14} {:>14} {:>14} {:>14}'.format(
        'Representation',
        'MAE Mean',
        'MAE Std',
        '1-MAE Mean',
        '1-MAE Std'
    )
)
print('-' * 94)

for name in probe_names:
    maes = np.array(
        [
            result['representations'][name]['best_mae']
            for result in seed_results
        ],
        dtype=np.float64
    )

    one_maes = 1.0 - maes

    print(
        '{:<22} {:>14.6f} {:>14.6f} {:>14.6f} {:>14.6f}'.format(
            name,
            maes.mean(),
            maes.std(ddof=1),
            one_maes.mean(),
            one_maes.std(ddof=1)
        )
    )

print('-' * 94)
print(
    'Global best mean-probe checkpoint: seed={}, epoch={}, mean MAE={:.6f}'.format(
        global_best_seed,
        global_best_epoch,
        global_best_mean_mae
    )
)
print('Saved probe checkpoint: {}'.format(probe_best_model_path))
print('=' * 94)


[2026-09-08 22:07:38,125]::3843851596::INFO::================ Probe Seed 1/1: 42 ================


---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=131


[2026-09-08 22:07:40,784]::3843851596::INFO::Backbone trainable params: 0
[2026-09-08 22:07:40,785]::3843851596::INFO::Probe trainable params: 72666
[2026-09-08 22:07:40,786]::3843851596::INFO::Probe dimensions: {'stage1_block1': 96, 'stage1_block2': 96, 'stage1': 192, 'stage2_block1': 192, 'stage2_block2': 192, 'stage2': 384, 'stage3_block1': 384, 'stage3_block2': 384, 'stage3_block3': 384, 'stage3_block4': 384, 'stage3_block5': 384, 'stage3_block6': 384, 'stage3': 768, 'stage4_block1': 768, 'stage4_block2': 768, 'stage4': 1536, 'final_video': 1536, 'final_cross': 1536}
Probe Seed 42 Train Epoch 1: 100%|█████████▉| 1497/1498 [02:01<00:00, 13.24it/s]/home/oem/anaconda3/envs/qx/lib/python3.10/site-packages/torch/nn/modules/loss.py:101: UserWarning: Using a target size (torch.Size([5])) that is different to the input size (torch.Size([1, 5])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction


Seed 42 | Epoch 001 | Mean Val MAE=0.116301 | Patience=0/10
Representation            Train MAE      Val MAE        1-MAE
--------------------------------------------------------------
stage1_block1              0.257288     0.151032     0.848968
stage1_block2              0.338215     0.170184     0.829816
stage1                     0.227835     0.171251     0.828749
stage2_block1              0.230380     0.150895     0.849105
stage2_block2              0.211819     0.155573     0.844427
stage2                     0.151717     0.124021     0.875979
stage3_block1              0.126040     0.108795     0.891205
stage3_block2              0.128182     0.104295     0.895705
stage3_block3              0.127450     0.105105     0.894895
stage3_block4              0.141028     0.102068     0.897932
stage3_block5              0.130244     0.109566     0.890434
stage3_block6              0.107657     0.101177     0.898823
stage3                     0.081909     0.095292     0.904708
stage4_b

Probe Seed 42 Val Epoch 2: 100%|██████████| 498/498 [00:46<00:00, 10.73it/s]


...

Probe Seed 42 Val Epoch 49: 100%|██████████| 498/498 [00:46<00:00, 10.66it/s]



Seed 42 | Epoch 049 | Mean Val MAE=0.090148 | Patience=6/10
Representation            Train MAE      Val MAE        1-MAE
--------------------------------------------------------------
stage1_block1              0.083398     0.099193     0.900807
stage1_block2              0.083525     0.099155     0.900845
stage1                     0.060206     0.093495     0.906505
stage2_block1              0.058982     0.093490     0.906510
stage2_block2              0.059904     0.093939     0.906061
stage2                     0.048951     0.089455     0.910545
stage3_block1              0.047534     0.088163     0.911837
stage3_block2              0.047358     0.088888     0.911112
stage3_block3              0.047083     0.088758     0.911242
stage3_block4              0.047086     0.089176     0.910824
stage3_block5              0.047001     0.088465     0.911535
stage3_block6              0.046860     0.087618     0.912382
stage3                     0.047033     0.088486     0.911514
stage4_b

Probe Seed 42 Val Epoch 50: 100%|██████████| 498/498 [00:45<00:00, 10.94it/s]


Seed 42 | Epoch 050 | Mean Val MAE=0.089324 | Patience=0/10
Representation            Train MAE      Val MAE        1-MAE
--------------------------------------------------------------
stage1_block1              0.083314     0.098995     0.901005
stage1_block2              0.083414     0.099134     0.900866
stage1                     0.060086     0.092458     0.907542
stage2_block1              0.058859     0.092956     0.907044
stage2_block2              0.059762     0.092712     0.907288
stage2                     0.048711     0.088753     0.911247
stage3_block1              0.047317     0.087381     0.912619
stage3_block2              0.047149     0.087627     0.912373
stage3_block3              0.046908     0.087869     0.912131
stage3_block4              0.046952     0.087613     0.912387
stage3_block5              0.046789     0.087260     0.912740
stage3_block6              0.046710     0.087116     0.912884
stage3                     0.047036     0.087541     0.912459
stage4_b


/home/oem/anaconda3/envs/qx/lib/python3.10/site-packages/numpy/core/_methods.py:265: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/oem/anaconda3/envs/qx/lib/python3.10/site-packages/numpy/core/_methods.py:257: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


In [15]:
# from torchinfo import summary

# summary(model, input_size = (4,15,4,3,224,224))